# Remove Missing Values and Export ML-Ready Dataset

This notebook keeps the existing cleaning and encoding logic, but in a smaller and easier-to-maintain form.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# REPLACE INPUT AND OUTPUT PATHS
INPUT_PATH = Path(r"YOUR PATH HERE")
ML_READY_OUTPUT_PATH = Path(r"YOUR PATH HERE")
YEAR_START = 2000
YEAR_END = 2026
REFERENCE_DATE = pd.Timestamp("2026-04-05")
COLUMNS_TO_DROP = [
    "id", "poster_path", "backdrop_path", "original_title", "homepage",
    "imdb_id", "recommendations", "merge_key", "record_origin", "actor_1", "actor_2", "actor_3",
    "actor_1_tmdb_name", "actor_1_tmdb_id", "actor_2_tmdb_name", "actor_2_tmdb_id",
    "actor_3_tmdb_name", "actor_3_tmdb_id", "credits",
]


def split_feature_types(frame: pd.DataFrame) -> dict[str, list[str]]:
    temporal = [column for column in frame.columns if pd.api.types.is_datetime64_any_dtype(frame[column])]
    binary = [
        column
        for column in frame.columns
        if pd.api.types.is_numeric_dtype(frame[column])
        and column not in temporal
        and set(frame[column].dropna().unique()).issubset({0, 1})
    ]
    numerical = [
        column
        for column in frame.columns
        if pd.api.types.is_numeric_dtype(frame[column]) and column not in temporal + binary
    ]
    categorical = [column for column in frame.columns if column not in temporal + numerical + binary]
    return {
        "temporal": temporal,
        "numerical": numerical,
        "binary": binary,
        "categorical": categorical,
        "ordinal": [],
        "nominal": categorical.copy(),
    }


def load_base_dataset(input_path: Path):
    if not input_path.exists():
        raise FileNotFoundError(f"Input dataset not found: {input_path}")

    df_raw = pd.read_csv(input_path, encoding="cp1252", low_memory=False)
    df_clean = df_raw.copy()
    df_clean[["revenue", "budget"]] = df_clean[["revenue", "budget"]].apply(pd.to_numeric, errors="coerce")

    invalid_core_rows = (
        df_clean["revenue"].isna()
        | (df_clean["revenue"] <= 0)
        | df_clean["budget"].isna()
        | (df_clean["budget"] <= 0)
    )
    df_clean = df_clean.loc[~invalid_core_rows].copy()

    X = df_clean.drop(columns=["revenue", *COLUMNS_TO_DROP], errors="ignore").copy()
    X["release_date"] = pd.to_datetime(X["release_date"], errors="coerce")
    y = df_clean["revenue"].copy()
    return df_raw, df_clean, X, y, invalid_core_rows, split_feature_types(X)


df_raw, df_clean, X, y, invalid_core_rows, feature_groups = load_base_dataset(INPUT_PATH)

print(f"Input dataset: {INPUT_PATH}")
print(f"Raw rows loaded: {len(df_raw)}")
print(f"Rows kept after budget/revenue validation: {len(X)}")
print(f"Feature count: {X.shape[1]}")

display(X.head(10))
display(y.head(10).to_frame(name="revenue"))

Input dataset: C:\Users\yohan\OneDrive\Documents\Movies-Dataset.csv
Raw rows loaded: 16441
Rows kept after budget/revenue validation: 16440
Feature count: 21


,title,genres,original_language,overview,popularity,production_companies,release_date,budget,runtime,status,...,vote_average,vote_count,keywords,adult,production_countries,spoken_languages,actor_1_popularity,actor_2_popularity,actor_3_popularity,mpaa_rating
0,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000,116,Released,...,7.079,1365,based on novel or book-sequel-kaiju,0.0,"China, United States of America",English,23.2858,0.0000,1.9394,PG-13
1,The Pope's Exorcist,Horror-Mystery-Thriller,en,Father Gabriele Amorth Chief Exorcist of the V...,5953.227,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,2023-04-05,18000000,103,Released,...,7.433,545,spain-rome italy-vatican-pope-pig-possession-c...,0.0,"Spain, United Kingdom, United States of America","English, Fulah, Spanish, Latin, German, Italian",4.5537,1.1310,1.0691,R
2,Transformers: Rise of the Beasts,Action-Adventure-Science Fiction,en,When a new threat capable of destroying the en...,5409.104,Skydance-Paramount-di Bonaventura Pictures-Bay...,2023-06-06,200000000,127,Released,...,7.340,1007,peru-alien-end of the world-based on cartoon-b...,0.0,United States of America,"Quechua, Spanish, English",2.2273,1.0588,2.2351,PG-13
3,Ant-Man and the Wasp: Quantumania,Action-Adventure-Science Fiction,en,Super-Hero partners Scott Lang and Hope van Dy...,4425.387,Marvel Studios-Kevin Feige Productions,2023-02-15,200000000,125,Released,...,6.507,2811,hero-ant-sequel-superhero-based on comic-famil...,0.0,United States of America,English,5.2578,3.8975,2.7409,PG-13
4,Creed III,Drama-Action,en,After dominating the boxing world Adonis Creed...,3994.342,Metro-Goldwyn-Mayer-Proximity Media-Balboa Pro...,2023-03-01,75000000,116,Released,...,7.262,1129,philadelphia pennsylvania-husband wife relatio...,0.0,United States of America,"English, Spanish",7.5815,4.0307,2.7409,PG-13
5,Insidious: The Red Door,Horror-Mystery-Thriller,en,To put their demons to rest once and for all J...,3512.648,Blumhouse Productions-Stage 6 Films-Screen Gem...,2023-07-05,16000000,107,Released,...,6.750,564,sequel-demon-franchise-insidious-supernatural ...,0.0,United States of America,English,1.6411,3.7455,0.4908,PG-13
6,Spider-Man: Across the Spider-Verse,Action-Adventure-Animation-Science Fiction,en,After reuniting with Gwen Stacy Brooklyn’s ful...,2550.738,Columbia Pictures-Sony Pictures Animation-Lord...,2023-05-31,100000000,140,Released,...,8.640,1684,sacrifice-villain-comic book-sequel-superhero-...,0.0,United States of America,"English, Hindi, Italian, Spanish",2.1644,7.0748,2.8530,PG
7,Shazam! Fury of the Gods,Action-Comedy-Fantasy-Adventure,en,Billy Batson and his foster siblings who trans...,2010.980,New Line Cinema-The Safran Company-DC Films,2023-03-15,125000000,130,Released,...,6.781,1602,superhero-end of the world-super power-aftercr...,0.0,United States of America,NaN,2.3596,1.2667,1.7115,PG-13
8,Knights of the Zodiac,Fantasy-Action-Adventure,en,When a headstrong street orphan Seiya in searc...,1882.774,Stage 6 Films-Toei Animation,2023-04-27,60000000,113,Released,...,6.561,471,superhero-based on manga-live action anime,0.0,"Japan, United States of America",English,5.1926,1.5591,0.7847,PG-13
9,Napoleon,History-War-Drama,en,An epic that details the checkered rise and fa...,1811.360,Scott Free Productions-Latina Pictures-Apple S...,2023-11-22,165000000,158,Released,...,6.516,1282,empire-france-biography-napoleon bonaparte-bas...,0.0,"Malta, United Kingdom, United States of America","French, English",3.7682,3.9274,2.4195,R


,revenue
0,352056482
1,65675816
2,407045464
3,475766228
4,269000000
5,175582093
6,512609552
7,133437105
8,6794519
9,213400000


In [3]:
feature_type_lookup = {
    **{column: "Temporal" for column in feature_groups["temporal"]},
    **{column: "Numerical" for column in feature_groups["numerical"]},
    **{column: "Binary" for column in feature_groups["binary"]},
    **{column: "Categorical (Ordinal)" for column in feature_groups["ordinal"]},
    **{column: "Categorical (Nominal)" for column in feature_groups["nominal"]},
}

feature_type_summary = pd.DataFrame(
    {
        "feature_type": list(feature_groups.keys()),
        "count": [len(columns) for columns in feature_groups.values()],
        "features": list(feature_groups.values()),
    }
)

validation_summary = pd.DataFrame(
    {
        "check": [
            "rows_removed_during_validation",
            "duplicate_rows_remaining",
            "duplicate_title_release_date_remaining",
            "release_date_missing_after_parse",
            f"release_date_before_{YEAR_START}-01-01",
            f"release_date_after_{REFERENCE_DATE.date()}",
        ],
        "count": [
            int(invalid_core_rows.sum()),
            int(df_clean.duplicated().sum()),
            int(df_clean.duplicated(subset=["title", "release_date"]).sum()),
            int(X["release_date"].isna().sum()),
            int((X["release_date"] < pd.Timestamp(f"{YEAR_START}-01-01")).sum()),
            int((X["release_date"] > REFERENCE_DATE).sum()),
        ],
    }
)

missing_values = (
    X.isna()
    .sum()
    .rename("Missing Count")
    .reset_index()
    .rename(columns={"index": "Feature"})
)
missing_values["Missing Ratio (%)"] = (missing_values["Missing Count"] / len(X) * 100).round(2)
missing_values["Feature Type"] = missing_values["Feature"].map(feature_type_lookup).fillna("Unknown")
missing_values = missing_values.sort_values(["Missing Count", "Feature"], ascending=[False, True]).reset_index(drop=True)

numeric_like_features = feature_groups["numerical"] + feature_groups["binary"]
numeric_data = X[numeric_like_features].apply(pd.to_numeric, errors="coerce")
numeric_alerts = pd.DataFrame(
    {
        "feature": numeric_data.columns,
        "missing_count": numeric_data.isna().sum().astype(int).to_list(),
        "zero_count": numeric_data.eq(0).sum().astype(int).to_list(),
        "negative_count": numeric_data.lt(0).sum().astype(int).to_list(),
        "p99_5": numeric_data.quantile(0.995).astype(float).to_list(),
        "max": numeric_data.max().astype(float).to_list(),
    }
).sort_values(["missing_count", "zero_count"], ascending=False)

vote_average_zero_with_zero_votes = int(
    ((numeric_data["vote_average"] == 0) & (numeric_data["vote_count"] == 0)).sum()
)

display(feature_type_summary)
display(validation_summary)
display(missing_values)
display(numeric_alerts)
print(f"Vote-average zeros paired with vote_count = 0: {vote_average_zero_with_zero_votes}")
print(f"Total missing values in feature set: {int(X.isna().sum().sum())}")

,feature_type,count,features
0,temporal,1,[release_date]
1,numerical,8,"[popularity, budget, runtime, vote_average, vo..."
2,binary,1,[adult]
3,categorical,11,"[title, genres, original_language, overview, p..."
4,ordinal,0,[]
5,nominal,11,"[title, genres, original_language, overview, p..."


,check,count
0,rows_removed_during_validation,1
1,duplicate_rows_remaining,0
2,duplicate_title_release_date_remaining,59
3,release_date_missing_after_parse,2663
4,release_date_before_2000-01-01,3661
5,release_date_after_2026-04-05,25


,Feature,Missing Count,Missing Ratio (%),Feature Type
0,mpaa_rating,8076,49.12,Categorical (Nominal)
1,actor_3_popularity,6429,39.11,Numerical
2,actor_2_popularity,6247,38.00,Numerical
3,actor_1_popularity,6038,36.73,Numerical
4,tagline,5118,31.13,Categorical (Nominal)
5,keywords,5092,30.97,Categorical (Nominal)
6,production_countries,4591,27.93,Categorical (Nominal)
7,production_companies,4121,25.07,Categorical (Nominal)
8,spoken_languages,3998,24.32,Categorical (Nominal)
9,release_date,2663,16.20,Temporal


,feature,missing_count,zero_count,negative_count,p99_5,max
7,actor_3_popularity,6429,176,0,1.047440e+01,2.374110e+01
6,actor_2_popularity,6247,211,0,1.464120e+01,3.290770e+01
5,actor_1_popularity,6038,284,0,2.357820e+01,3.290770e+01
8,adult,557,15597,0,1.000000e+00,1.000000e+00
3,vote_average,0,5107,0,1.000000e+01,1.000000e+01
4,vote_count,0,5099,0,1.706328e+04,3.485700e+04
2,runtime,0,2333,0,2.838050e+02,9.990000e+02
0,popularity,0,1653,0,2.943929e+02,8.763998e+03
1,budget,0,0,0,2.050000e+08,5.000000e+09


Vote-average zeros paired with vote_count = 0: 5095
Total missing values in feature set: 57677


In [4]:
def build_ml_ready_dataset(features: pd.DataFrame, target: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    movies_model_df = features.copy()
    movies_model_df["revenue"] = target.to_numpy()

    release_year = movies_model_df["release_date"].dt.year
    complete_case_mask = movies_model_df.notna().all(axis=1)
    year_range_mask = release_year.between(YEAR_START, YEAR_END, inclusive="both")
    final_keep_mask = complete_case_mask & year_range_mask

    movies_ml_ready = (
        movies_model_df.loc[final_keep_mask]
        .sort_values("release_date")
        .reset_index(drop=True)
    )

    cleaning_summary = pd.DataFrame(
        {
            "metric": [
                "rows_before_complete_case_filter",
                "rows_removed_for_any_missing_value",
                f"rows_removed_for_year_outside_{YEAR_START}_{YEAR_END}",
                "rows_removed_for_both_conditions",
                "rows_removed_total",
                "rows_kept_final",
                "total_missing_values_after_cleaning",
                "minimum_release_year_after_cleaning",
                "maximum_release_year_after_cleaning",
            ],
            "value": [
                int(len(movies_model_df)),
                int(((~complete_case_mask) & year_range_mask).sum()),
                int((complete_case_mask & (~year_range_mask)).sum()),
                int(((~complete_case_mask) & (~year_range_mask)).sum()),
                int((~final_keep_mask).sum()),
                int(len(movies_ml_ready)),
                int(movies_ml_ready.isna().sum().sum()),
                int(movies_ml_ready["release_date"].dt.year.min()),
                int(movies_ml_ready["release_date"].dt.year.max()),
            ],
        }
    )
    return movies_ml_ready, cleaning_summary


movies_ml_ready, cleaning_summary = build_ml_ready_dataset(X, y)
ML_READY_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
movies_ml_ready.to_csv(ML_READY_OUTPUT_PATH, index=False, encoding="utf-8-sig")

display(cleaning_summary)
display(movies_ml_ready.head(10))
print(f"Saved ML-ready dataset to: {ML_READY_OUTPUT_PATH}")

X = movies_ml_ready.drop(columns=["revenue"]).copy()
y = movies_ml_ready["revenue"].copy()

,metric,value
0,rows_before_complete_case_filter,16440
1,rows_removed_for_any_missing_value,6247
2,rows_removed_for_year_outside_2000_2026,2463
3,rows_removed_for_both_conditions,3874
4,rows_removed_total,12584
5,rows_kept_final,3856
6,total_missing_values_after_cleaning,0
7,minimum_release_year_after_cleaning,2000
8,maximum_release_year_after_cleaning,2024


,title,genres,original_language,overview,popularity,production_companies,release_date,budget,runtime,status,...,vote_count,keywords,adult,production_countries,spoken_languages,actor_1_popularity,actor_2_popularity,actor_3_popularity,mpaa_rating,revenue
0,My Dog Skip,Comedy-Drama-Family,en,A shy boy is unable to make friends in Yazoo C...,9.050,Warner Bros. Pictures-Alcon Entertainment,2000-01-12,6000000,95,Released,...,239,mississippi river-dog-pets,0.0,United States of America,English,1.9447,5.4331,0.7548,PG,35547761
1,Next Friday,Comedy,en,A streetwise man flees South Central Los Angel...,17.608,New Line Cinema,2000-01-12,11000000,98,Released,...,475,prison-repayment-gang war-boy gang-revenge-pri...,0.0,United States of America,English,3.2183,2.5073,0.9141,R,59827328
2,Supernova,Science Fiction-Horror-Fantasy-Thriller,en,Set in the 22nd century when a battered salvag...,11.103,Hammerhead Productions-Screenland Pictures-Uni...,2000-01-14,90000000,91,Released,...,315,black people-starship-future-supernova-blast,0.0,"Switzerland, United States of America",English,5.5735,3.2632,3.0546,PG-13,14828081
3,Ring 0,Horror-Thriller,ja,Taking place thirty years before the events of...,9.197,Ring 0 Production Group,2000-01-22,6000000,99,Released,...,205,journalist-dying and death-theatre group-prequ...,0.0,Japan,Japanese,2.2732,1.7178,3.4217,NR,11918069
4,Isn't She Great,Comedy-Drama,de,An unsuccessful over-the-top actress becomes a...,2.743,Universal Pictures-Mutual Film Company-Toho-BB...,2000-01-28,36000000,95,Released,...,30,autism,0.0,United States of America,English,1.8918,2.0412,3.0844,R,3003296
5,The Broken Hearts Club: A Romantic Comedy,Comedy-Drama-Romance,en,A close-knit group of gay friends share the em...,6.647,Banner Entertainment-Meanwhile Films,2000-02-01,1000000,96,Released,...,68,roommate-baseball-male friendship-in vitro fer...,0.0,United States of America,English,4.2960,2.6312,0.7111,R,1744858
6,The Beach,Drama-Adventure-Romance-Thriller,en,Twenty-something Richard travels to Thailand a...,15.389,Figment Films,2000-02-03,40000000,119,Released,...,3866,hippie-based on novel or book-exotic island-be...,0.0,"United States of America, United Kingdom","English, French, Croatian, Swedish, Thai",13.1864,1.5672,1.0605,R,144056873
7,Scream 3,Horror-Mystery,en,While Sidney lives in safely guarded seclusion...,80.174,Craven-Maddalena Films-Dimension Films-Konrad ...,2000-02-03,40000000,116,Released,...,2820,movie business-isolation-mask-ex-cop-serial ki...,0.0,United States of America,English,2.5795,5.4998,4.6881,R,161834276
8,Gun Shy,Action-Comedy-Romance-Thriller,en,Legendary undercover DEA agent Charlie Mayough...,7.221,Fortis Films-Hollywood Pictures,2000-02-04,10000000,101,Released,...,110,drug cartel-nervous breakdown,0.0,United States of America,English,7.7290,10.0451,2.9687,R,1631839
9,The Million Dollar Hotel,Drama-Thriller,en,The Million Dollar Hotel starts with a jump fr...,9.043,Kintop Pictures,2000-02-09,8000000,122,Released,...,270,hotel-fbi-confidence-junkie-friendship-insanit...,0.0,"Germany, United Kingdom, United States of America",English,7.9623,7.3130,2.5570,R,105983


Saved ML-ready dataset to: C:\Users\yohan\OneDrive\Documents\Movies-Dataset-no-missing-2000-2026-ML-Ready.csv


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

NUMERICAL_FEATURES = [
    "budget",
    "runtime",
    "actor_1_popularity",
    "actor_2_popularity",
    "actor_3_popularity",
    "revenue",
]
MULTILABEL_FEATURES = [
    "production_countries",
    "production_companies",
    "genres",
    "spoken_languages",
]
ONEHOT_FEATURES = ["original_language", "mpaa_rating"]
TEXT_FEATURES = ["keywords", "tagline", "overview"]
PASSTHROUGH_FEATURES = ["title", "release_date", *NUMERICAL_FEATURES]

PROTECTED_COMPANY_NAMES = [
    "Metro-Goldwyn-Mayer",
    "Craven-Maddalena Films",
    "Davis-Panzer Productions",
    "Tele M?nchen Fernseh Produktionsgesellschaft (TMG)",
]

MULTILABEL_FEATURE_CONFIGS = {
    "production_countries": {
        "column": "production_countries",
        "separator": ",",
        "prefix": "country_",
        "top_k": 10,
        "protected_values": None,
    },
    "production_companies": {
        "column": "production_companies",
        "separator": "-",
        "prefix": "studio_",
        "top_k": 20,
        "protected_values": PROTECTED_COMPANY_NAMES,
    },
    "genres": {
        "column": "genres",
        "separator": "-",
        "prefix": "genre_",
        "top_k": None,
        "protected_values": None,
    },
    "spoken_languages": {
        "column": "spoken_languages",
        "separator": ",",
        "prefix": "spoken_language_",
        "top_k": 5,
        "protected_values": None,
    },
}
ONEHOT_FEATURE_CONFIGS = {column: {"column": column} for column in ONEHOT_FEATURES}
ENCODED_OUTPUT_PATH = ML_READY_OUTPUT_PATH.parent / f"{ML_READY_OUTPUT_PATH.stem}-encoded.csv"


def ensure_series(X, column, fill_value=""):
    if isinstance(X, pd.DataFrame):
        series = X[column]
    elif isinstance(X, pd.Series):
        series = X
    else:
        series = pd.Series(np.asarray(X).ravel(), name=column)
    return series.fillna(fill_value).astype(str)


class MultiLabelColumnEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column, separator, prefix, top_k=None, protected_values=None):
        self.column = column
        self.separator = separator
        self.prefix = prefix
        self.top_k = top_k
        self.protected_values = protected_values
        self._placeholder = "__PROTECTED_HYPHEN__"

    def _prepare_labels(self, X):
        return ensure_series(X, self.column).map(self._split_labels)

    def _split_labels(self, value):
        value = value.strip()
        if not value:
            return []

        for protected in self.protected_values or []:
            value = value.replace(protected, protected.replace("-", self._placeholder))

        labels = []
        for item in value.split(self.separator):
            cleaned = item.strip().replace(self._placeholder, "-")
            if cleaned:
                labels.append(cleaned)
        return labels

    def fit(self, X, y=None):
        labels = self._prepare_labels(X)
        self.encoder_ = MultiLabelBinarizer()
        encoded = pd.DataFrame(
            self.encoder_.fit_transform(labels),
            columns=self.encoder_.classes_,
            index=labels.index,
        )
        label_counts = encoded.sum()
        self.selected_labels_ = (
            label_counts.nlargest(self.top_k).index.tolist()
            if self.top_k is not None
            else label_counts.index.tolist()
        )
        self.feature_names_ = [f"{self.prefix}{label}" for label in self.selected_labels_]
        return self

    def transform(self, X):
        labels = self._prepare_labels(X)
        encoded = pd.DataFrame(
            self.encoder_.transform(labels),
            columns=self.encoder_.classes_,
            index=labels.index,
        )
        encoded = encoded.reindex(columns=self.selected_labels_, fill_value=0)
        encoded.columns = self.feature_names_
        return encoded

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


class OneHotColumnEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column

    def _prepare_series(self, X):
        series = ensure_series(X, self.column, fill_value="Unknown").str.strip()
        return series.replace("", "Unknown")

    def fit(self, X, y=None):
        series = self._prepare_series(X)
        self.encoder_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=int)
        self.encoder_.fit(series.to_frame(name=self.column))
        self.feature_names_ = self.encoder_.get_feature_names_out([self.column]).tolist()
        return self

    def transform(self, X):
        series = self._prepare_series(X)
        encoded = self.encoder_.transform(series.to_frame(name=self.column))
        return pd.DataFrame(encoded, columns=self.feature_names_, index=series.index)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


class MovieTextEncoder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        keyword_column="keywords",
        tagline_column="tagline",
        overview_column="overview",
        keyword_top_k=25,
        tfidf_max_features=500,
        n_components=5,
        random_state=42,
    ):
        self.keyword_column = keyword_column
        self.tagline_column = tagline_column
        self.overview_column = overview_column
        self.keyword_top_k = keyword_top_k
        self.tfidf_max_features = tfidf_max_features
        self.n_components = n_components
        self.random_state = random_state

    def _get_frame(self, X):
        columns = [self.keyword_column, self.tagline_column, self.overview_column]
        if isinstance(X, pd.DataFrame):
            return X.loc[:, columns].copy()
        return pd.DataFrame(X, columns=columns)

    def _prepare_keywords(self, frame):
        keywords = frame[self.keyword_column].fillna("").astype(str)
        keywords = keywords.where(keywords.ne("nan"), "")
        return keywords.str.replace("-", " ", regex=False)

    def _prepare_text_columns(self, frame):
        tagline = frame[self.tagline_column].fillna("").astype(str)
        overview = frame[self.overview_column].fillna("").astype(str)
        return tagline, overview

    def fit(self, X, y=None):
        frame = self._get_frame(X)
        keywords = self._prepare_keywords(frame)
        tagline, overview = self._prepare_text_columns(frame)
        combined_text = tagline + " " + overview

        self.keyword_vectorizer_ = CountVectorizer(max_features=self.keyword_top_k, stop_words="english")
        self.keyword_vectorizer_.fit(keywords)
        self.keyword_feature_names_ = [
            f"kw_{feature_name}"
            for feature_name in self.keyword_vectorizer_.get_feature_names_out()
        ]

        self.tfidf_vectorizer_ = TfidfVectorizer(stop_words="english", max_features=self.tfidf_max_features)
        text_tfidf_matrix = self.tfidf_vectorizer_.fit_transform(combined_text)

        self.svd_ = TruncatedSVD(n_components=self.n_components, random_state=self.random_state)
        self.svd_.fit(text_tfidf_matrix)
        self.theme_feature_names_ = [f"text_theme_{index + 1}" for index in range(self.n_components)]
        self.feature_names_ = [
            "num_keywords",
            "overview_word_count",
            "tagline_word_count",
            *self.keyword_feature_names_,
            *self.theme_feature_names_,
        ]
        return self

    def transform(self, X):
        frame = self._get_frame(X)
        keywords = self._prepare_keywords(frame)
        tagline, overview = self._prepare_text_columns(frame)
        combined_text = tagline + " " + overview

        meta_features = pd.DataFrame(
            {
                "num_keywords": keywords.str.split().str.len(),
                "overview_word_count": overview.str.split().str.len(),
                "tagline_word_count": tagline.str.split().str.len(),
            },
            index=frame.index,
        )
        keyword_features = pd.DataFrame(
            self.keyword_vectorizer_.transform(keywords).toarray(),
            columns=self.keyword_feature_names_,
            index=frame.index,
        )
        theme_features = pd.DataFrame(
            self.svd_.transform(self.tfidf_vectorizer_.transform(combined_text)),
            columns=self.theme_feature_names_,
            index=frame.index,
        )
        return pd.concat([meta_features, keyword_features, theme_features], axis=1)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


def build_preprocessor():
    transformers = [("base_features", "passthrough", PASSTHROUGH_FEATURES)]
    transformers.extend(
        (column, Pipeline([("encoder", MultiLabelColumnEncoder(**MULTILABEL_FEATURE_CONFIGS[column]))]), [column])
        for column in MULTILABEL_FEATURES
    )
    transformers.extend(
        (column, Pipeline([("encoder", OneHotColumnEncoder(**ONEHOT_FEATURE_CONFIGS[column]))]), [column])
        for column in ONEHOT_FEATURES
    )
    transformers.append(("text_features", Pipeline([("encoder", MovieTextEncoder())]), TEXT_FEATURES))
    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_feature_summary(preprocessor):
    summary_rows = []
    selected_features = {}

    for name, transformer, _ in preprocessor.transformers_:
        if name == "base_features" or not hasattr(transformer, "named_steps"):
            continue

        feature_names = transformer.named_steps["encoder"].get_feature_names_out().tolist()
        selected_features[name] = feature_names

        if name in MULTILABEL_FEATURES:
            encoding_name = "MultiLabelBinarizer"
        elif name in ONEHOT_FEATURES:
            encoding_name = "OneHotEncoder"
        else:
            encoding_name = "CountVectorizer + TfidfVectorizer + TruncatedSVD"

        summary_rows.append(
            {
                "feature": name,
                "encoding": encoding_name,
                "selected_columns": len(feature_names),
                "sample_columns": ", ".join(feature_names[:5]),
            }
        )

    return selected_features, pd.DataFrame(summary_rows)

In [6]:
df = pd.read_csv(ML_READY_OUTPUT_PATH)

feature_pipeline = Pipeline(steps=[("preprocessor", build_preprocessor())])
feature_pipeline.set_output(transform="pandas")

df_encoded = feature_pipeline.fit_transform(df)
selected_features, encoding_summary_df = build_feature_summary(
    feature_pipeline.named_steps["preprocessor"]
)

df_encoded.to_csv(ENCODED_OUTPUT_PATH, index=False)
print(f"Saved encoded dataset to: {ENCODED_OUTPUT_PATH}")

Saved encoded dataset to: C:\Users\yohan\OneDrive\Documents\Movies-Dataset-no-missing-2000-2026-ML-Ready-encoded.csv


## ML Pipeline

In [7]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(ENCODED_OUTPUT_PATH)

TARGET_COLUMN = "revenue"
EXCLUDED_COLUMNS = ["title", "release_date", TARGET_COLUMN]
RANDOM_STATE = 42
TEST_SIZE = 0.20

model_features = [column for column in df.columns if column not in EXCLUDED_COLUMNS]
binary_features = [
    column
    for column in model_features
    if df[column].dropna().isin([0, 1]).all()
]
continuous_features = [column for column in model_features if column not in binary_features]

X = df[model_features].copy()
y = df[TARGET_COLUMN].astype(float).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

best_params = {
    "preprocessor__continuous__scaler": "passthrough",
    "regressor__fit_intercept": True,
    "regressor__positive": False,
}

continuous_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),   # placeholder; replaced by best_params
])

binary_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

model = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("continuous", continuous_pipe, continuous_features),
        ("binary", binary_pipe, binary_features),
    ])),
    ("regressor", LinearRegression()),
])

model.set_params(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

X_train


,budget,runtime,actor_1_popularity,actor_2_popularity,actor_3_popularity,country_United States of America,country_United Kingdom,country_France,country_Germany,country_Canada,country_Japan,country_China,country_Australia,country_India,country_Spain,studio_Universal Pictures,studio_Warner Bros. Pictures,studio_Columbia Pictures,studio_20th Century Fox,studio_Paramount,studio_New Line Cinema,studio_Walt Disney Pictures,studio_Lionsgate,studio_Relativity Media,studio_Metro-Goldwyn-Mayer,studio_DreamWorks Pictures,studio_Canal+,studio_Summit Entertainment,studio_StudioCanal,studio_Village Roadshow Pictures,studio_Miramax,studio_Touchstone Pictures,studio_Screen Gems,studio_Dune Entertainment,studio_Blumhouse Productions,genre_Action,genre_Adventure,genre_Animation,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Family,genre_Fantasy,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science Fiction,genre_TV Movie,genre_Thriller,genre_War,genre_Western,spoken_language_English,spoken_language_Spanish,spoken_language_French,spoken_language_German,spoken_language_Russian,original_language_af,original_language_ca,original_language_cn,original_language_da,original_language_de,original_language_el,original_language_en,original_language_es,original_language_fa,original_language_fi,original_language_fr,original_language_he,original_language_hi,original_language_hu,original_language_id,original_language_it,original_language_ja,original_language_kn,original_language_ko,original_language_ml,original_language_nl,original_language_no,original_language_pl,original_language_pt,original_language_ro,original_language_ru,original_language_sv,original_language_ta,original_language_te,original_language_th,original_language_tl,original_language_tr,original_language_ur,original_language_zh,mpaa_rating_G,mpaa_rating_NC-17,mpaa_rating_NR,mpaa_rating_PG,mpaa_rating_PG-13,mpaa_rating_R,num_keywords,overview_word_count,tagline_word_count,kw_aftercreditsstinger,kw_based,kw_book,kw_child,kw_city,kw_death,kw_director,kw_duringcreditsstinger,kw_family,kw_father,kw_friendship,kw_love,kw_murder,kw_new,kw_novel,kw_police,kw_relationship,kw_school,kw_sequel,kw_story,kw_true,kw_war,kw_woman,kw_world,kw_york,text_theme_1,text_theme_2,text_theme_3,text_theme_4,text_theme_5
2462,5000000,117,6.0604,4.4178,4.1557,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,18,27,9,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0.257469,0.180568,0.158203,0.004072,0.068851
798,6400000,107,0.8874,0.7596,0.9529,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,21,73,6,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0.214432,-0.037220,0.011587,-0.009140,0.052228
3427,21000000,101,2.3099,0.8703,4.3457,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,29,45,6,0,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,3,0,0,0,0,0,1,0,0,0.259616,-0.100981,-0.162534,0.183676,-0.043332
2338,3500000,135,3.8597,0.9105,0.9785,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,8,20,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.099474,-0.029120,-0.032838,0.009419,0.001156
2266,3800000,116,4.6211,0.8268,0.3781,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,11,45,4,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0.315093,0.198902,-0.185166,-0.148957,-0.056534
...,...,...,...,...,...,...,...,...,...,.

### Results and Diagnostics
This section fits the final Linear Regression pipeline once using the best parameters found earlier. There is no hyperparameter tuning, no K-fold cross-validation, and no search here.


In [8]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score

alt.data_transformers.disable_max_rows()


def evaluate_regression_split(actual, predicted):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "MAE": float(mean_absolute_error(actual, predicted)),
        "MedianAE": float(median_absolute_error(actual, predicted)),
        "R2": float(r2_score(actual, predicted)),
    }


train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

train_metrics = evaluate_regression_split(y_train, train_predictions)
test_metrics = evaluate_regression_split(y_test, test_predictions)

metrics_summary_df = pd.DataFrame(
    [
        {"Split": "Train", **train_metrics},
        {"Split": "Test", **test_metrics},
    ]
)
error_metric_long_df = metrics_summary_df.melt(
    id_vars="Split",
    value_vars=["RMSE", "MAE", "MedianAE"],
    var_name="Metric",
    value_name="Value",
)

best_params_display_df = pd.DataFrame(
    {
        "Parameter": list(best_params.keys()),
        "Value": [str(value) for value in best_params.values()],
    }
)

run_summary_df = pd.DataFrame(
    {
        "Metric": [
            "Training rows",
            "Testing rows",
            "Total features used",
            "Continuous features",
            "Binary features",
            "Train share",
            "Test share",
        ],
        "Value": [
            int(len(X_train)),
            int(len(X_test)),
            int(len(model_features)),
            int(len(continuous_features)),
            int(len(binary_features)),
            f"{1 - TEST_SIZE:.0%}",
            f"{TEST_SIZE:.0%}",
        ],
    }
)

test_results_df = pd.DataFrame(
    {
        "title": df.loc[y_test.index, "title"] if "title" in df.columns else pd.Series(index=y_test.index, dtype="object"),
        "release_date": df.loc[y_test.index, "release_date"] if "release_date" in df.columns else pd.Series(index=y_test.index, dtype="object"),
        "actual_revenue": y_test,
        "predicted_revenue": test_predictions,
    }
).reset_index(drop=True)

test_results_df["residual"] = test_results_df["actual_revenue"] - test_results_df["predicted_revenue"]
test_results_df["absolute_error"] = test_results_df["residual"].abs()
test_results_df["absolute_pct_error"] = np.where(
    test_results_df["actual_revenue"].ne(0),
    test_results_df["absolute_error"] / test_results_df["actual_revenue"],
    np.nan,
)
test_results_df["release_date"] = pd.to_datetime(test_results_df["release_date"], errors="coerce").dt.strftime("%Y-%m-%d")

metrics_display_df = metrics_summary_df.copy()
for column in ["RMSE", "MAE", "MedianAE"]:
    metrics_display_df[column] = metrics_display_df[column].map(lambda value: f"${value:,.0f}")
metrics_display_df["R2"] = metrics_display_df["R2"].map(lambda value: f"{value:.3f}")

top_error_display_df = (
    test_results_df
    .sort_values("absolute_error", ascending=False)
    .head(10)
    .copy()
)
for column in ["actual_revenue", "predicted_revenue", "residual", "absolute_error"]:
    top_error_display_df[column] = top_error_display_df[column].map(lambda value: f"${value:,.0f}")
top_error_display_df["absolute_pct_error"] = top_error_display_df["absolute_pct_error"].map(
    lambda value: f"{value:.1%}" if pd.notna(value) else ""
)

rmse_gap = test_metrics["RMSE"] - train_metrics["RMSE"]
summary_lines = [
    "- Final model: **Linear Regression** with the best parameters from the tuning notebook.",
    f"- Test RMSE: **${test_metrics['RMSE']:,.0f}** | Test MAE: **${test_metrics['MAE']:,.0f}** | Test R2: **{test_metrics['R2']:.3f}**.",
    f"- Generalization gap: test RMSE is **${abs(rmse_gap):,.0f}** {'higher' if rmse_gap >= 0 else 'lower'} than train RMSE.",
]
summary_text = chr(10).join(summary_lines)

display(Markdown("### Final Model Snapshot"))
display(Markdown(summary_text))

display(Markdown("### Run Summary"))
display(run_summary_df)

display(Markdown("### Best Parameters Used"))
display(best_params_display_df)

display(Markdown("### Train vs Test Metrics"))
display(metrics_display_df)

display(Markdown("### Largest Test Errors"))
display(top_error_display_df)

error_chart = alt.Chart(error_metric_long_df).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=alt.X("Metric:N", title=None, sort=["RMSE", "MAE", "MedianAE"]),
    xOffset=alt.XOffset("Split:N"),
    y=alt.Y("Value:Q", title="Revenue"),
    color=alt.Color("Split:N", scale=alt.Scale(range=["#1f77b4", "#ff7f0e"])),
    tooltip=[
        alt.Tooltip("Split:N", title="Split"),
        alt.Tooltip("Metric:N", title="Metric"),
        alt.Tooltip("Value:Q", title="Value", format=",.0f"),
    ],
).properties(title="Train vs Test Error Metrics", width=280, height=320)

r2_floor = min(-0.1, metrics_summary_df["R2"].min() - 0.05)
r2_chart = alt.Chart(metrics_summary_df).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4, color="#6a4c93").encode(
    x=alt.X("Split:N", title=None),
    y=alt.Y("R2:Q", title="R2", scale=alt.Scale(domain=[r2_floor, 1.0])),
    tooltip=[
        alt.Tooltip("Split:N", title="Split"),
        alt.Tooltip("R2:Q", title="R2", format=".3f"),
    ],
).properties(title="Train vs Test R2", width=180, height=320)

line_min = float(min(test_results_df["actual_revenue"].min(), test_results_df["predicted_revenue"].min()))
line_max = float(max(test_results_df["actual_revenue"].max(), test_results_df["predicted_revenue"].max()))
reference_line_df = pd.DataFrame({"value": [line_min, line_max]})

actual_vs_predicted_chart = (
    alt.Chart(test_results_df)
    .mark_circle(size=70, opacity=0.70, color="#1f77b4")
    .encode(
        x=alt.X("actual_revenue:Q", title="Actual revenue"),
        y=alt.Y("predicted_revenue:Q", title="Predicted revenue"),
        tooltip=[
            alt.Tooltip("title:N", title="Title"),
            alt.Tooltip("release_date:N", title="Release date"),
            alt.Tooltip("actual_revenue:Q", title="Actual", format=",.0f"),
            alt.Tooltip("predicted_revenue:Q", title="Predicted", format=",.0f"),
            alt.Tooltip("absolute_error:Q", title="Absolute error", format=",.0f"),
        ],
    )
    .properties(title="Actual vs Predicted Revenue (Test)", width=330, height=320)
)
reference_line_chart = alt.Chart(reference_line_df).mark_line(color="#d62728", strokeDash=[6, 4]).encode(
    x="value:Q",
    y="value:Q",
)

zero_rule_df = pd.DataFrame({"zero": [0]})
residual_chart = (
    alt.Chart(test_results_df)
    .mark_circle(size=70, opacity=0.70, color="#ff7f0e")
    .encode(
        x=alt.X("predicted_revenue:Q", title="Predicted revenue"),
        y=alt.Y("residual:Q", title="Residual"),
        tooltip=[
            alt.Tooltip("title:N", title="Title"),
            alt.Tooltip("release_date:N", title="Release date"),
            alt.Tooltip("predicted_revenue:Q", title="Predicted", format=",.0f"),
            alt.Tooltip("residual:Q", title="Residual", format=",.0f"),
            alt.Tooltip("absolute_error:Q", title="Absolute error", format=",.0f"),
        ],
    )
    .properties(title="Residuals vs Predicted Revenue", width=330, height=320)
)
zero_rule_chart = alt.Chart(zero_rule_df).mark_rule(color="#d62728", strokeDash=[6, 4]).encode(y="zero:Q")

residual_distribution_chart = alt.Chart(test_results_df).mark_bar(color="#2ca02c").encode(
    x=alt.X("residual:Q", bin=alt.Bin(maxbins=30), title="Residual"),
    y=alt.Y("count():Q", title="Movie count"),
    tooltip=[alt.Tooltip("count():Q", title="Movie count")],
).properties(title="Residual Distribution", width=330, height=320)

display((error_chart | r2_chart | (actual_vs_predicted_chart + reference_line_chart)).resolve_scale(y="independent"))
display(((residual_chart + zero_rule_chart) | residual_distribution_chart).resolve_scale(y="independent"))


### Final Model Snapshot

- Final model: **Linear Regression** with the best parameters from the tuning notebook.
- Test RMSE: **$121,262,351** | Test MAE: **$76,260,145** | Test R2: **0.563**.
- Generalization gap: test RMSE is **$15,800,198** lower than train RMSE.

### Run Summary

,Metric,Value
0,Training rows,3084
1,Testing rows,772
2,Total features used,132
3,Continuous features,35
4,Binary features,97
5,Train share,80%
6,Test share,20%


### Best Parameters Used

...

,Split,RMSE,MAE,MedianAE,R2
0,Train,"$137,062,549","$76,461,031","$44,734,598",0.619
1,Test,"$121,262,351","$76,260,145","$47,542,780",0.563


### Largest Test Errors

,title,release_date,actual_revenue,predicted_revenue,residual,absolute_error,absolute_pct_error
553,The Super Mario Bros. Movie,2023-04-05,"$1,362,000,000","$370,316,941","$991,683,059","$991,683,059",72.8%
465,Minions,2015-06-17,"$1,156,730,962","$296,539,934","$860,191,028","$860,191,028",74.4%
232,Black Panther,2018-02-13,"$1,349,926,083","$666,251,412","$683,674,671","$683,674,671",50.6%
434,Superman Returns,2006-06-28,"$391,081,192","$954,323,204","$-563,242,012","$563,242,012",144.0%
241,Iron Man 3,2013-04-18,"$1,215,577,205","$667,906,148","$547,671,057","$547,671,057",45.1%
270,Argylle,2024-01-31,"$96,175,599","$641,942,028","$-545,766,429","$545,766,429",567.5%
395,Zootopia,2016-02-11,"$1,023,784,195","$498,443,051","$525,341,144","$525,341,144",51.3%
743,Turning Red,2022-03-10,"$21,328,962","$515,332,026","$-494,003,064","$494,003,064",2316.1%
738,Aladdin,2019-05-22,"$1,054,304,000","$565,613,338","$488,690,662","$488,690,662",46.4%
760,Fast X,2023-05-17,"$704,709,660","$1,168,803,131","$-464,093,471","$464,093,471",65.9%


alt.HConcatChart(...)

alt.HConcatChart(...)

### Predict Revenue From User Input
Edit the values in the dictionary below, run the cell, and the notebook will transform the raw movie fields into model-ready features before predicting revenue.


In [9]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

if "feature_pipeline" not in globals() or "model" not in globals():
    raise RuntimeError("Run the feature-engineering cells and the ML Pipeline cells before running this prediction cell.")

USER_MOVIE_INPUT = {
    "title": "My New Movie",
    "release_date": "2026-07-15",
    "budget": 120_000_000,
    "runtime": 125,
    "actor_1_popularity": 35.0,
    "actor_2_popularity": 22.0,
    "actor_3_popularity": 14.0,
    "production_countries": "United States of America,Canada",
    "production_companies": "Warner Bros. Pictures-Legendary Pictures",
    "genres": "Action-Adventure-Science Fiction",
    "spoken_languages": "English,Spanish",
    "original_language": "en",
    "mpaa_rating": "PG-13",
    "keywords": "hero rescue future mission",
    "tagline": "The future needs one more chance.",
    "overview": "A veteran pilot and a young scientist team up to stop a global catastrophe before it destroys the world.",
}

RAW_INPUT_COLUMNS = [
    "title",
    "release_date",
    "budget",
    "runtime",
    "actor_1_popularity",
    "actor_2_popularity",
    "actor_3_popularity",
    "production_countries",
    "production_companies",
    "genres",
    "spoken_languages",
    "original_language",
    "mpaa_rating",
    "keywords",
    "tagline",
    "overview",
]
NUMERIC_INPUT_COLUMNS = [
    "budget",
    "runtime",
    "actor_1_popularity",
    "actor_2_popularity",
    "actor_3_popularity",
]


def build_user_movie_frame(movie_input: dict) -> pd.DataFrame:
    user_frame = pd.DataFrame([movie_input]).copy()

    for column in RAW_INPUT_COLUMNS:
        if column not in user_frame.columns:
            user_frame[column] = np.nan if column in NUMERIC_INPUT_COLUMNS else ""

    for column in NUMERIC_INPUT_COLUMNS:
        user_frame[column] = pd.to_numeric(user_frame[column], errors="coerce")

    text_like_columns = [column for column in RAW_INPUT_COLUMNS if column not in NUMERIC_INPUT_COLUMNS]
    for column in text_like_columns:
        user_frame[column] = user_frame[column].fillna("").astype(str)

    user_frame["revenue"] = 0.0
    return user_frame[RAW_INPUT_COLUMNS + ["revenue"]]


user_movie_raw_df = build_user_movie_frame(USER_MOVIE_INPUT)
user_movie_encoded_df = feature_pipeline.transform(user_movie_raw_df)
user_model_input = user_movie_encoded_df.reindex(columns=model_features, fill_value=0)

predicted_revenue = float(model.predict(user_model_input)[0])

user_input_display_df = user_movie_raw_df[RAW_INPUT_COLUMNS].T.reset_index()
user_input_display_df.columns = ["Field", "Value"]
prediction_summary_df = pd.DataFrame(
    {
        "Output": ["Predicted revenue"],
        "Value": [f"${predicted_revenue:,.0f}"],
    }
)

display(Markdown("### Entered Movie Values"))
display(user_input_display_df)

display(Markdown("### Prediction"))
display(prediction_summary_df)

print(f"Predicted revenue: ${predicted_revenue:,.2f}")


### Entered Movie Values

,Field,Value
0,title,My New Movie
1,release_date,2026-07-15
2,budget,120000000
3,runtime,125
4,actor_1_popularity,35.0
5,actor_2_popularity,22.0
6,actor_3_popularity,14.0
7,production_countries,"United States of America,Canada"
8,production_companies,Warner Bros. Pictures-Legendary Pictures
9,genres,Action-Adventure-Science Fiction


### Prediction

,Output,Value
0,Predicted revenue,"$415,947,848"


Predicted revenue: $415,947,848.32
